In [1]:
import os
import numpy as np
import pandas as pd

BASE_DIR = '/Users/francescoceron/Desktop/data'
patients = range(23)
T = 797

# Variabile risposta
Y = np.zeros(T, dtype=int)
Y[701:] = 1

rows = []

metrics = [d for d in os.listdir(BASE_DIR)
           if os.path.isdir(os.path.join(BASE_DIR, d))]

print("Metriche trovate:", metrics)

for p in patients:
    data_patient = {}

    for m in metrics:
        folder = os.path.join(BASE_DIR, m)

        if m == 'n1cut':
            filename = f'n1cut_{p}.npy'
        else:
            filename = f'{m}_patient_{p}.npy'

        file_path = os.path.join(folder, filename)

        if not os.path.exists(file_path):
            raise FileNotFoundError(file_path)

        data_patient[m] = np.load(file_path)

    for t in range(T):
        row = {
            'patient': p,
            'time': t,
            'Y': Y[t]
        }
        for m in metrics:
            row[m] = data_patient[m][t]

        rows.append(row)

df = pd.DataFrame(rows)

print(df.shape)
print(df.head())


Metriche trovate: ['n1cut', 'betweenness', 'pagerank', 'clustering', 'modularity', 'weighted_degree']
(18331, 9)
   patient  time  Y     n1cut  betweenness  pagerank  clustering  modularity  \
0        0     0  0  0.764749     0.008614  0.006135    0.636981    0.139320   
1        0     1  0  0.750125     0.008194  0.006135    0.615737    0.126293   
2        0     2  0  0.773472     0.007932  0.006135    0.681328    0.129237   
3        0     3  0  0.779494     0.007981  0.006135    0.673039    0.115107   
4        0     4  0  0.562642     0.008882  0.006135    0.612838    0.116401   

   weighted_degree  
0         5.719314  
1         6.229029  
2         6.966947  
3         6.578285  
4         5.959365  


In [2]:
# dimensioni corrette
assert df.groupby('patient').size().nunique() == 1

# sanity check ictal/pre
print(df.groupby('Y').size())

print(df[['betweenness','n1cut']].describe())


Y
0    16123
1     2208
dtype: int64
        betweenness         n1cut
count  18331.000000  18331.000000
mean       0.009149      0.400654
std        0.002558      0.184454
min        0.005137      0.000000
25%        0.007828      0.259324
50%        0.008586      0.395554
75%        0.009552      0.537187
max        0.040453      1.000000


In [3]:


# ---------- BASIC CHECK ----------
print("Shape dataframe:", df.shape)
print("\nDistribuzione Y:")
print(df['Y'].value_counts())

# ---------- METRIC LIST ----------
feature_cols = [c for c in df.columns if c not in ['patient', 'time', 'Y']]
print("\nFeature usate:", feature_cols)

# ---------- PRE vs ICTAL STATS ----------
print("\n===== PRE-ICTAL vs ICTAL (GLOBAL STATS) =====")

stats_rows = []

for f in feature_cols:
    pre = df[df['Y'] == 0][f]
    ict = df[df['Y'] == 1][f]

    mean_pre, std_pre = pre.mean(), pre.std()
    mean_ict, std_ict = ict.mean(), ict.std()

    pooled_std = np.sqrt((std_pre**2 + std_ict**2) / 2)
    effect_size = (mean_ict - mean_pre) / pooled_std if pooled_std > 0 else np.nan

    stats_rows.append([
        f, mean_pre, std_pre, mean_ict, std_ict, effect_size
    ])

stats_df = pd.DataFrame(
    stats_rows,
    columns=['feature', 'mean_pre', 'std_pre', 'mean_ict', 'std_ict', 'effect_size']
)

print(stats_df.sort_values('effect_size', key=np.abs, ascending=False))

# ---------- CORRELATION MATRIX ----------
print("\n===== FEATURE CORRELATION (GLOBAL) =====")
corr = df[feature_cols].corr()
print(corr)

# ---------- PER-PATIENT VARIANCE ----------
print("\n===== VARIANZA PER PAZIENTE =====")

var_rows = []

for p in df['patient'].unique():
    sub = df[df['patient'] == p]
    for f in feature_cols:
        var_rows.append([p, f, sub[f].var()])

var_df = pd.DataFrame(var_rows, columns=['patient', 'feature', 'variance'])

print(
    var_df
    .groupby('feature')['variance']
    .describe()[['mean', 'std', 'min', 'max']]
)

# ---------- ICTAL WINDOW CHECK ----------
print("\n===== ICTAL WINDOW CHECK (ultime 97 finestre) =====")
ictal_window = df[(df['time'] >= 701) & (df['time'] <= 797)]
print("Numero finestre ictali:", len(ictal_window))


Shape dataframe: (18331, 9)

Distribuzione Y:
Y
0    16123
1     2208
Name: count, dtype: int64

Feature usate: ['n1cut', 'betweenness', 'pagerank', 'clustering', 'modularity', 'weighted_degree']

===== PRE-ICTAL vs ICTAL (GLOBAL STATS) =====
           feature  mean_pre   std_pre  mean_ict   std_ict   effect_size
4       modularity  0.173763  0.038429  0.220522  0.083114  7.221501e-01
0            n1cut  0.416682  0.176078  0.283616  0.201290 -7.036605e-01
1      betweenness  0.008946  0.002323  0.010632  0.003526  5.645723e-01
5  weighted_degree  7.869765  2.549028  8.382581  2.628748  1.980598e-01
3       clustering  0.542480  0.077847  0.532166  0.074939 -1.349920e-01
2         pagerank  0.005577  0.001433  0.005577  0.001434 -2.363362e-08

===== FEATURE CORRELATION (GLOBAL) =====
                    n1cut  betweenness  pagerank  clustering  modularity  \
n1cut            1.000000    -0.383941 -0.095167    0.287761   -0.584665   
betweenness     -0.383941     1.000000  0.843426   -

In [4]:


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score

# =========================
# PARAMETRI
# =========================
PATIENTS = sorted(df['patient'].unique())
ONSET = 701
N_WIN = 97  # finestre per classe

# feature selezionate (decise dall’analisi)
FEATURES = ['modularity', 'n1cut', 'betweenness']

results = []

# =========================
# LOOP LOPO
# =========================
for test_p in PATIENTS:

    # ---------- split pazienti ----------
    train_df = df[df['patient'] != test_p].copy()
    test_df  = df[df['patient'] == test_p].copy()

    # ---------- TRAIN SET (bilanciato) ----------
    # ictal
    train_ictal = train_df[train_df['Y'] == 1]

    # pre-ictal IMMEDIATO
    train_pre = train_df[
        (train_df['Y'] == 0) &
        (train_df['time'] >= ONSET - N_WIN) &
        (train_df['time'] < ONSET)
    ]

    # controllo sicurezza
    if len(train_ictal) < N_WIN or len(train_pre) < N_WIN:
        raise ValueError("Non abbastanza finestre per il train")

    train_ictal = train_ictal.sample(N_WIN, random_state=42)
    train_pre   = train_pre.sample(N_WIN, random_state=42)

    train_bal = pd.concat([train_pre, train_ictal])

    X_train = train_bal[FEATURES].values
    y_train = train_bal['Y'].values

    # ---------- TEST SET ----------
    X_test = test_df[FEATURES].values
    y_test = test_df['Y'].values

    # ---------- standardizzazione ----------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # ---------- modello ----------
    clf = LogisticRegression(
        penalty='l2',
        solver='liblinear',
        class_weight='balanced',
        random_state=42
    )

    clf.fit(X_train, y_train)

    # ---------- predizioni ----------
    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # ---------- metriche ----------
    auc = roc_auc_score(y_test, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    results.append({
        'patient': test_p,
        'AUC': auc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'balanced_accuracy': bal_acc
    })

    print(f"Paziente {test_p:02d} | AUC={auc:.3f} | "
          f"Sens={sensitivity:.3f} | Spec={specificity:.3f}")

# =========================
# RISULTATI FINALI
# =========================
res_df = pd.DataFrame(results)

print("\n===== RISULTATI GLOBALI (LOPO) =====")
print(res_df.describe()[['AUC', 'sensitivity', 'specificity', 'balanced_accuracy']])

print("\nAUC medio:", res_df['AUC'].mean())
print("Sensitivity media:", res_df['sensitivity'].mean())
print("Specificity media:", res_df['specificity'].mean())


Paziente 00 | AUC=0.884 | Sens=0.646 | Spec=0.981
Paziente 01 | AUC=0.308 | Sens=0.562 | Spec=0.218
Paziente 02 | AUC=0.779 | Sens=0.604 | Spec=0.756
Paziente 03 | AUC=0.690 | Sens=0.396 | Spec=0.969
Paziente 04 | AUC=0.961 | Sens=0.896 | Spec=0.892
Paziente 05 | AUC=0.746 | Sens=0.417 | Spec=0.877
Paziente 06 | AUC=0.497 | Sens=0.469 | Spec=0.880
Paziente 07 | AUC=0.590 | Sens=0.365 | Spec=0.766
Paziente 08 | AUC=0.527 | Sens=0.333 | Spec=0.799
Paziente 09 | AUC=0.864 | Sens=0.854 | Spec=0.715
Paziente 10 | AUC=0.475 | Sens=0.271 | Spec=0.859
Paziente 11 | AUC=0.983 | Sens=0.958 | Spec=0.927
Paziente 12 | AUC=0.825 | Sens=0.667 | Spec=0.967
Paziente 13 | AUC=0.930 | Sens=0.719 | Spec=0.970
Paziente 14 | AUC=0.512 | Sens=0.240 | Spec=0.760
Paziente 15 | AUC=0.855 | Sens=1.000 | Spec=0.000
Paziente 16 | AUC=1.000 | Sens=1.000 | Spec=0.916
Paziente 17 | AUC=0.902 | Sens=0.802 | Spec=0.827
Paziente 18 | AUC=0.832 | Sens=0.885 | Spec=0.672
Paziente 19 | AUC=0.826 | Sens=0.625 | Spec=0.960


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

# Assumiamo df già definito con colonne: ['patient','time','Y',metrics...]

patients = df['patient'].unique()
features = ['n1cut','betweenness','pagerank','clustering','modularity','weighted_degree']

# Funzione per calcolare metriche
def compute_metrics(y_true, y_pred_prob, threshold=0.5):
    y_pred = (y_pred_prob >= threshold).astype(int)
    auc = roc_auc_score(y_true, y_pred_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    acc = accuracy_score(y_true, y_pred)
    return auc, sens, spec, acc

# Container risultati
results_logistic = []
results_gb = []

for p in patients:
    # Train-test split LOPO
    train_df = df[df['patient'] != p]
    test_df = df[df['patient'] == p]

    X_train = train_df[features].values
    y_train = train_df['Y'].values
    X_test = test_df[features].values
    y_test = test_df['Y'].values

    # Logistic Regression
    model_log = LogisticRegression(max_iter=1000)
    model_log.fit(X_train, y_train)
    y_prob_log = model_log.predict_proba(X_test)[:,1]
    auc_log, sens_log, spec_log, acc_log = compute_metrics(y_test, y_prob_log)
    results_logistic.append({
        'patient': p, 'AUC': auc_log, 'sensitivity': sens_log,
        'specificity': spec_log, 'accuracy': acc_log
    })

    # Gradient Boosting
    model_gb = GradientBoostingClassifier(n_estimators=200, max_depth=3)
    model_gb.fit(X_train, y_train)
    y_prob_gb = model_gb.predict_proba(X_test)[:,1]
    auc_gb, sens_gb, spec_gb, acc_gb = compute_metrics(y_test, y_prob_gb)
    results_gb.append({
        'patient': p, 'AUC': auc_gb, 'sensitivity': sens_gb,
        'specificity': spec_gb, 'accuracy': acc_gb
    })

# Converti in dataframe
df_log = pd.DataFrame(results_logistic)
df_gb = pd.DataFrame(results_gb)

print("===== Logistic Regression per paziente =====")
print(df_log)
print("\nAUC medio:", df_log['AUC'].mean())
print("Accuracy media:", df_log['accuracy'].mean())
print("Balanced accuracy media:", ((df_log['sensitivity'] + df_log['specificity'])/2).mean())

print("\n===== Gradient Boosting per paziente =====")
print(df_gb)
print("\nAUC medio:", df_gb['AUC'].mean())
print("Accuracy media:", df_gb['accuracy'].mean())
print("Balanced accuracy media:", ((df_gb['sensitivity'] + df_gb['specificity'])/2).mean())

# --- OPTIONAL: confronto finestre pre-ictal immediate vs random ---
N_PRE = 97  # numero finestre pre-ictal da confrontare
results_random = []

for p in patients:
    test_df = df[df['patient'] == p]
    pre_idx = test_df.index[test_df['Y']==0].to_numpy()
    ictal_idx = test_df.index[test_df['Y']==1].to_numpy()

    # Random sampling di N_PRE finestre pre-ictal
    np.random.seed(123)
    sampled_idx = np.random.choice(pre_idx, N_PRE, replace=False)
    sample_idx = np.concatenate([sampled_idx, ictal_idx])
    y_test_sample = df.loc[sample_idx, 'Y'].values
    X_test_sample = df.loc[sample_idx, features].values

    X_train = df[df['patient'] != p][features].values
    y_train = df[df['patient'] != p]['Y'].values

    model_log = LogisticRegression(max_iter=1000)
    model_log.fit(X_train, y_train)
    y_prob_sample = model_log.predict_proba(X_test_sample)[:,1]
    auc_r, sens_r, spec_r, acc_r = compute_metrics(y_test_sample, y_prob_sample)
    results_random.append({'patient': p, 'AUC': auc_r, 'sensitivity': sens_r,
                           'specificity': spec_r, 'accuracy': acc_r})

df_random = pd.DataFrame(results_random)
print("\n===== Logistic Regression con finestre pre-ictal random =====")
print(df_random)
print("\nAUC medio random:", df_random['AUC'].mean())
print("Accuracy media random:", df_random['accuracy'].mean())
print("Balanced accuracy media random:", ((df_random['sensitivity'] + df_random['specificity'])/2).mean())


===== Logistic Regression per paziente =====
    patient       AUC  sensitivity  specificity  accuracy
0         0  0.914111     0.322917     1.000000  0.918444
1         1  0.321564     0.010417     0.971469  0.855709
2         2  0.496761     0.333333     0.974322  0.897114
3         3  0.847004     0.041667     1.000000  0.884567
4         4  0.949120     0.479167     1.000000  0.937265
5         5  0.641598     0.000000     1.000000  0.879548
6         6  0.637794     0.000000     1.000000  0.879548
7         7  0.418985     0.020833     0.994294  0.877039
8         8  0.538650     0.187500     1.000000  0.902133
9         9  0.802648     0.416667     0.978602  0.910916
10       10  0.722405     0.041667     1.000000  0.884567
11       11  0.933191     0.000000     1.000000  0.879548
12       12  0.923755     0.156250     1.000000  0.898369
13       13  0.795307     0.385417     0.997147  0.923463
14       14  0.679461     0.000000     1.000000  0.879548
15       15  0.827865     0

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

# Parametri lag
MAX_LAG = 5

# Pazienti e metriche già definiti
all_metrics = ['n1cut', 'betweenness', 'pagerank', 'clustering', 'modularity', 'weighted_degree']
patients = df['patient'].unique()
results = []

for p in patients:
    df_p = df[df['patient']==p].copy()
    # Normalizzazione per paziente
    scaler = StandardScaler()
    df_p[all_metrics] = scaler.fit_transform(df_p[all_metrics])
    
    # Creazione lag features
    for lag in range(1, MAX_LAG+1):
        for m in all_metrics:
            df_p[f'{m}_lag{lag}'] = df_p[m].shift(lag)
    df_p.dropna(inplace=True)  # rimuove le prime righe senza lag

    X = df_p[[c for c in df_p.columns if c not in ['patient','time','Y']]]
    Y = df_p['Y'].values

    # LOPO: train = tutti tranne p, test = p
    X_train, Y_train = [], []
    X_test, Y_test = X.values, Y
    for q in patients:
        if q != p:
            df_q = df[df['patient']==q].copy()
            # Normalizzazione per paziente
            df_q[all_metrics] = scaler.fit_transform(df_q[all_metrics])
            # Lag features
            for lag in range(1, MAX_LAG+1):
                for m in all_metrics:
                    df_q[f'{m}_lag{lag}'] = df_q[m].shift(lag)
            df_q.dropna(inplace=True)
            X_train.append(df_q[[c for c in df_q.columns if c not in ['patient','time','Y']]].values)
            Y_train.append(df_q['Y'].values)
    X_train = np.vstack(X_train)
    Y_train = np.hstack(Y_train)
    
    # Bilanciamento: downsample pre-ictal per migliorare sensibilità
    idx_ictal = np.where(Y_train==1)[0]
    idx_preictal = np.where(Y_train==0)[0]
    n_ictal = len(idx_ictal)
    np.random.seed(123)
    idx_preictal_down = np.random.choice(idx_preictal, n_ictal, replace=False)
    selected_idx = np.hstack([idx_ictal, idx_preictal_down])
    X_train_bal = X_train[selected_idx]
    Y_train_bal = Y_train[selected_idx]

    # Logistic Regression
    clf_lr = LogisticRegression(max_iter=1000)
    clf_lr.fit(X_train_bal, Y_train_bal)
    Y_pred_lr = clf_lr.predict(X_test)
    Y_prob_lr = clf_lr.predict_proba(X_test)[:,1]

    # Gradient Boosting
    clf_gb = GradientBoostingClassifier(n_estimators=200, max_depth=3)
    clf_gb.fit(X_train_bal, Y_train_bal)
    Y_pred_gb = clf_gb.predict(X_test)
    Y_prob_gb = clf_gb.predict_proba(X_test)[:,1]

    # Metriche
    def compute_metrics(Y_true, Y_pred, Y_prob):
        auc = roc_auc_score(Y_true, Y_prob)
        acc = accuracy_score(Y_true, Y_pred)
        bal_acc = balanced_accuracy_score(Y_true, Y_pred)
        tn, fp, fn, tp = confusion_matrix(Y_true, Y_pred).ravel()
        sens = tp/(tp+fn) if (tp+fn)>0 else 0
        spec = tn/(tn+fp) if (tn+fp)>0 else 0
        return auc, acc, bal_acc, sens, spec

    auc_lr, acc_lr, bal_lr, sens_lr, spec_lr = compute_metrics(Y_test, Y_pred_lr, Y_prob_lr)
    auc_gb, acc_gb, bal_gb, sens_gb, spec_gb = compute_metrics(Y_test, Y_pred_gb, Y_prob_gb)

    results.append({
        'patient': p,
        'model':'LogisticRegression',
        'AUC': auc_lr,
        'accuracy': acc_lr,
        'balanced_accuracy': bal_lr,
        'sensitivity': sens_lr,
        'specificity': spec_lr
    })
    results.append({
        'patient': p,
        'model':'GradientBoosting',
        'AUC': auc_gb,
        'accuracy': acc_gb,
        'balanced_accuracy': bal_gb,
        'sensitivity': sens_gb,
        'specificity': spec_gb
    })

df_results = pd.DataFrame(results)
print(df_results.groupby('model')[['AUC','accuracy','balanced_accuracy','sensitivity','specificity']].mean())


                         AUC  accuracy  balanced_accuracy  sensitivity  \
model                                                                    
GradientBoosting    0.887068  0.875274           0.827329     0.764040   
LogisticRegression  0.772140  0.817249           0.741215     0.640851   

                    specificity  
model                            
GradientBoosting       0.890617  
LogisticRegression     0.841579  


In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score


In [13]:
def get_balanced_data(df, patient_test):
    # split train / test
    train_df = df[df['patient'] != patient_test]
    test_df  = df[df['patient'] == patient_test]

    # --- TRAIN: 97 ictal + 97 pre-ictal ---
    ictal_train = train_df[train_df['Y'] == 1]
    pre_train   = train_df[train_df['Y'] == 0].sample(
        n=len(ictal_train), random_state=42
    )

    train_bal = pd.concat([ictal_train, pre_train])

    # --- TEST: tutte le finestre del paziente ---
    return train_bal, test_df


In [14]:
feature_cols = [c for c in df.columns if c not in ['patient', 'time', 'Y']]

auc_scores = []
bal_acc_scores = []

for test_patient in df['patient'].unique():

    train_bal, test_df = get_balanced_data(df, test_patient)

    X_train = train_bal[feature_cols].values
    y_train = train_bal['Y'].values

    X_test  = test_df[feature_cols].values
    y_test  = test_df['Y'].values

    # --- Patient-wise normalization ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # --- Random Forest (REGOLARIZZATA) ---
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=10,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    )

    rf.fit(X_train, y_train)

    y_prob = rf.predict_proba(X_test)[:, 1]

    auc_scores.append(roc_auc_score(y_test, y_prob))
    bal_acc_scores.append(
        balanced_accuracy_score(y_test, y_prob > 0.5)
    )

print("RF AUC medio:", np.mean(auc_scores))
print("RF Balanced Acc medio:", np.mean(bal_acc_scores))
print


RF AUC medio: 0.7569016958175692
RF Balanced Acc medio: 0.6870535621989291


In [15]:
from hmmlearn.hmm import GaussianHMM


In [16]:
def train_hmm(train_df, feature_cols, n_states=3):

    X = train_df[feature_cols].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    hmm = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=300,
        random_state=42
    )

    hmm.fit(X)

    return hmm, scaler


In [17]:
def map_states_to_labels(hmm, train_df, scaler, feature_cols):

    X = scaler.transform(train_df[feature_cols].values)
    states = hmm.predict(X)

    train_df = train_df.copy()
    train_df['state'] = states

    # % di ictal per stato
    state_ictal_rate = (
        train_df
        .groupby('state')['Y']
        .mean()
    )

    # stato più ictal-like
    ictal_state = state_ictal_rate.idxmax()

    return ictal_state


In [18]:
auc_scores = []
bal_acc_scores = []

for test_patient in df['patient'].unique():

    train_bal, test_df = get_balanced_data(df, test_patient)

    # --- Train HMM ---
    hmm, scaler = train_hmm(train_bal, feature_cols, n_states=3)

    ictal_state = map_states_to_labels(
        hmm, train_bal, scaler, feature_cols
    )

    # --- Test ---
    X_test = scaler.transform(test_df[feature_cols].values)
    test_states = hmm.predict(X_test)

    # prob = 1 se stato ictal-like
    y_prob = (test_states == ictal_state).astype(float)

    auc_scores.append(roc_auc_score(test_df['Y'], y_prob))
    bal_acc_scores.append(
        balanced_accuracy_score(test_df['Y'], y_prob)
    )

print("HMM AUC medio:", np.mean(auc_scores))
print("HMM Balanced Acc medio:", np.mean(bal_acc_scores))


HMM AUC medio: 0.7062432808203603
HMM Balanced Acc medio: 0.7062432808203603


In [19]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)

rf_results = []

for test_patient in df['patient'].unique():

    train_bal, test_df = get_balanced_data(df, test_patient)

    X_train = train_bal[feature_cols].values
    y_train = train_bal['Y'].values

    X_test  = test_df[feature_cols].values
    y_test  = test_df['Y'].values

    # --- Normalizzazione patient-wise ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=10,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    )

    rf.fit(X_train, y_train)

    y_prob = rf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)

    rf_results.append({
        'patient': test_patient,
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob)
    })

rf_results_df = pd.DataFrame(rf_results)

print(rf_results_df)
print("\nRF – MEDIE:")
print(rf_results_df[['accuracy','balanced_accuracy','auc']].mean())


    patient  accuracy  balanced_accuracy       auc
0         0  0.956085           0.898620  0.923086
1         1  0.294856           0.338430  0.276101
2         2  0.616060           0.628908  0.649533
3         3  0.903388           0.621434  0.885610
4         4  0.941029           0.912536  0.986582
5         5  0.897114           0.658323  0.830302
6         6  0.838143           0.669750  0.889622
7         7  0.711418           0.584225  0.520239
8         8  0.432873           0.412394  0.392772
9         9  0.786700           0.793338  0.817612
10       10  0.816813           0.572218  0.719954
11       11  0.968632           0.973178  0.993506
12       12  0.925972           0.841046  0.885758
13       13  0.947302           0.844181  0.776956
14       14  0.818068           0.680813  0.798636
15       15  0.124216           0.502140  0.895373
16       16  0.994981           0.997147  0.999837
17       17  0.794228           0.815598  0.879176
18       18  0.713927          

In [20]:
hmm_results = []

for test_patient in df['patient'].unique():

    train_bal, test_df = get_balanced_data(df, test_patient)

    # --- Train HMM ---
    hmm, scaler = train_hmm(train_bal, feature_cols, n_states=3)

    ictal_state = map_states_to_labels(
        hmm, train_bal, scaler, feature_cols
    )

    # --- Test ---
    X_test = scaler.transform(test_df[feature_cols].values)
    test_states = hmm.predict(X_test)

    y_prob = (test_states == ictal_state).astype(float)
    y_pred = y_prob.astype(int)
    y_test = test_df['Y'].values

    hmm_results.append({
        'patient': test_patient,
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob)
    })

hmm_results_df = pd.DataFrame(hmm_results)

print(hmm_results_df)
print("\nHMM – MEDIE:")
print(hmm_results_df[['accuracy','balanced_accuracy','auc']].mean())


    patient  accuracy  balanced_accuracy       auc
0         0  0.961104           0.838542  0.838542
1         1  0.314931           0.295902  0.295902
2         2  0.662484           0.668784  0.668784
3         3  0.922208           0.677083  0.677083
4         4  0.984944           0.955480  0.955480
5         5  0.887077           0.576201  0.576201
6         6  0.913425           0.690071  0.690071
7         7  0.872020           0.635067  0.635067
8         8  0.888331           0.626360  0.626360
9         9  0.844417           0.803673  0.803673
10       10  0.885822           0.597963  0.597963
11       11  0.997491           0.989583  0.989583
12       12  0.930991           0.713542  0.713542
13       13  0.954831           0.852956  0.852956
14       14  0.834379           0.762007  0.762007
15       15  0.922208           0.681578  0.681578
16       16  0.998745           0.999287  0.999287
17       17  0.913425           0.658605  0.658605
18       18  0.838143          

# Escludi patient

In [21]:
excluded_patients = [1, 20, 22]

patients_to_evaluate = [
    p for p in df['patient'].unique()
    if p not in excluded_patients
]


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

rf_results = []

for test_patient in patients_to_evaluate:

    train_bal, test_df = get_balanced_data(df, test_patient)

    X_train = train_bal[feature_cols].values
    y_train = train_bal['Y'].values

    X_test  = test_df[feature_cols].values
    y_test  = test_df['Y'].values

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=10,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42
    )

    rf.fit(X_train, y_train)

    y_prob = rf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)

    rf_results.append({
        'patient': test_patient,
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob)
    })

rf_results_df = pd.DataFrame(rf_results)

print("RANDOM FOREST – risultati per paziente (senza 1,20,22)")
print(rf_results_df)
print("\nRF – MEDIE:")
print(rf_results_df[['accuracy','balanced_accuracy','auc']].mean())


RANDOM FOREST – risultati per paziente (senza 1,20,22)
    patient  accuracy  balanced_accuracy       auc
0         0  0.956085           0.898620  0.923086
1         2  0.616060           0.628908  0.649533
2         3  0.903388           0.621434  0.885610
3         4  0.941029           0.912536  0.986582
4         5  0.897114           0.658323  0.830302
5         6  0.838143           0.669750  0.889622
6         7  0.711418           0.584225  0.520239
7         8  0.432873           0.412394  0.392772
8         9  0.786700           0.793338  0.817612
9        10  0.816813           0.572218  0.719954
10       11  0.968632           0.973178  0.993506
11       12  0.925972           0.841046  0.885758
12       13  0.947302           0.844181  0.776956
13       14  0.818068           0.680813  0.798636
14       15  0.124216           0.502140  0.895373
15       16  0.994981           0.997147  0.999837
16       17  0.794228           0.815598  0.879176
17       18  0.713927      

In [23]:
from hmmlearn.hmm import GaussianHMM

hmm_results = []

for test_patient in patients_to_evaluate:

    train_bal, test_df = get_balanced_data(df, test_patient)

    hmm, scaler = train_hmm(
        train_bal,
        feature_cols,
        n_states=3
    )

    ictal_state = map_states_to_labels(
        hmm,
        train_bal,
        scaler,
        feature_cols
    )

    X_test = scaler.transform(test_df[feature_cols].values)
    test_states = hmm.predict(X_test)

    y_prob = (test_states == ictal_state).astype(float)
    y_pred = y_prob.astype(int)
    y_test = test_df['Y'].values

    hmm_results.append({
        'patient': test_patient,
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_prob)
    })

hmm_results_df = pd.DataFrame(hmm_results)

print("HMM – risultati per paziente (senza 1,20,22)")
print(hmm_results_df)
print("\nHMM – MEDIE:")
print(hmm_results_df[['accuracy','balanced_accuracy','auc']].mean())


HMM – risultati per paziente (senza 1,20,22)
    patient  accuracy  balanced_accuracy       auc
0         0  0.961104           0.838542  0.838542
1         2  0.662484           0.668784  0.668784
2         3  0.922208           0.677083  0.677083
3         4  0.984944           0.955480  0.955480
4         5  0.887077           0.576201  0.576201
5         6  0.913425           0.690071  0.690071
6         7  0.872020           0.635067  0.635067
7         8  0.888331           0.626360  0.626360
8         9  0.844417           0.803673  0.803673
9        10  0.885822           0.597963  0.597963
10       11  0.997491           0.989583  0.989583
11       12  0.930991           0.713542  0.713542
12       13  0.954831           0.852956  0.852956
13       14  0.834379           0.762007  0.762007
14       15  0.922208           0.681578  0.681578
15       16  0.998745           0.999287  0.999287
16       17  0.913425           0.658605  0.658605
17       18  0.838143           0.863

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

# Parametri lag
MAX_LAG = 5

# Pazienti e metriche già definiti
all_metrics = ['n1cut', 'betweenness', 'pagerank', 'clustering', 'modularity', 'weighted_degree']
patients = df['patient'].unique()
results = []

for p in patients:
    df_p = df[df['patient']==p].copy()
    # Normalizzazione per paziente
    scaler = StandardScaler()
    df_p[all_metrics] = scaler.fit_transform(df_p[all_metrics])
    
    # Creazione lag features
    for lag in range(1, MAX_LAG+1):
        for m in all_metrics:
            df_p[f'{m}_lag{lag}'] = df_p[m].shift(lag)
    df_p.dropna(inplace=True)  # rimuove le prime righe senza lag

    X = df_p[[c for c in df_p.columns if c not in ['patient','time','Y']]]
    Y = df_p['Y'].values

    # LOPO: train = tutti tranne p, test = p
    X_train, Y_train = [], []
    X_test, Y_test = X.values, Y
    for q in patients:
        if q != p:
            df_q = df[df['patient']==q].copy()
            # Normalizzazione per paziente
            df_q[all_metrics] = scaler.fit_transform(df_q[all_metrics])
            # Lag features
            for lag in range(1, MAX_LAG+1):
                for m in all_metrics:
                    df_q[f'{m}_lag{lag}'] = df_q[m].shift(lag)
            df_q.dropna(inplace=True)
            X_train.append(df_q[[c for c in df_q.columns if c not in ['patient','time','Y']]].values)
            Y_train.append(df_q['Y'].values)
    X_train = np.vstack(X_train)
    Y_train = np.hstack(Y_train)
    
    # Bilanciamento: downsample pre-ictal per migliorare sensibilità
    idx_ictal = np.where(Y_train==1)[0]
    idx_preictal = np.where(Y_train==0)[0]
    n_ictal = len(idx_ictal)
    np.random.seed(123)
    idx_preictal_down = np.random.choice(idx_preictal, n_ictal, replace=False)
    selected_idx = np.hstack([idx_ictal, idx_preictal_down])
    X_train_bal = X_train[selected_idx]
    Y_train_bal = Y_train[selected_idx]

    # Logistic Regression
    clf_lr = LogisticRegression(max_iter=1000)
    clf_lr.fit(X_train_bal, Y_train_bal)
    Y_pred_lr = clf_lr.predict(X_test)
    Y_prob_lr = clf_lr.predict_proba(X_test)[:,1]

    # Gradient Boosting
    clf_gb = GradientBoostingClassifier(n_estimators=200, max_depth=3)
    clf_gb.fit(X_train_bal, Y_train_bal)
    Y_pred_gb = clf_gb.predict(X_test)
    Y_prob_gb = clf_gb.predict_proba(X_test)[:,1]

    # Metriche
    def compute_metrics(Y_true, Y_pred, Y_prob):
        auc = roc_auc_score(Y_true, Y_prob)
        acc = accuracy_score(Y_true, Y_pred)
        bal_acc = balanced_accuracy_score(Y_true, Y_pred)
        tn, fp, fn, tp = confusion_matrix(Y_true, Y_pred).ravel()
        sens = tp/(tp+fn) if (tp+fn)>0 else 0
        spec = tn/(tn+fp) if (tn+fp)>0 else 0
        return auc, acc, bal_acc, sens, spec

    auc_lr, acc_lr, bal_lr, sens_lr, spec_lr = compute_metrics(Y_test, Y_pred_lr, Y_prob_lr)
    auc_gb, acc_gb, bal_gb, sens_gb, spec_gb = compute_metrics(Y_test, Y_pred_gb, Y_prob_gb)

    results.append({
        'patient': p,
        'model':'LogisticRegression',
        'AUC': auc_lr,
        'accuracy': acc_lr,
        'balanced_accuracy': bal_lr,
        'sensitivity': sens_lr,
        'specificity': spec_lr
    })
    results.append({
        'patient': p,
        'model':'GradientBoosting',
        'AUC': auc_gb,
        'accuracy': acc_gb,
        'balanced_accuracy': bal_gb,
        'sensitivity': sens_gb,
        'specificity': spec_gb
    })

df_results = pd.DataFrame(results)
print(df_results.groupby('model')[['AUC','accuracy','balanced_accuracy','sensitivity','specificity']].mean())


In [24]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
)

MAX_LAG = 5
all_metrics = ['n1cut', 'betweenness', 'pagerank', 'clustering', 'modularity', 'weighted_degree']

excluded_patients = [1, 20, 22]
patients = [p for p in df['patient'].unique() if p not in excluded_patients]

results = []

for p in patients:

    # ---------- TEST PATIENT ----------
    df_p = df[df['patient'] == p].copy()

    scaler_p = StandardScaler()
    df_p[all_metrics] = scaler_p.fit_transform(df_p[all_metrics])

    for lag in range(1, MAX_LAG + 1):
        for m in all_metrics:
            df_p[f'{m}_lag{lag}'] = df_p[m].shift(lag)

    df_p.dropna(inplace=True)

    X_test = df_p[[c for c in df_p.columns if c not in ['patient', 'time', 'Y']]].values
    Y_test = df_p['Y'].values

    # ---------- TRAIN SET ----------
    X_train, Y_train = [], []

    for q in df['patient'].unique():
        if q != p:
            df_q = df[df['patient'] == q].copy()

            scaler_q = StandardScaler()
            df_q[all_metrics] = scaler_q.fit_transform(df_q[all_metrics])

            for lag in range(1, MAX_LAG + 1):
                for m in all_metrics:
                    df_q[f'{m}_lag{lag}'] = df_q[m].shift(lag)

            df_q.dropna(inplace=True)

            X_train.append(
                df_q[[c for c in df_q.columns if c not in ['patient', 'time', 'Y']]].values
            )
            Y_train.append(df_q['Y'].values)

    X_train = np.vstack(X_train)
    Y_train = np.hstack(Y_train)

    # ---------- BILANCIAMENTO 97 vs 97 ----------
    idx_ictal = np.where(Y_train == 1)[0]
    idx_pre   = np.where(Y_train == 0)[0]

    np.random.seed(123)
    idx_pre_down = np.random.choice(idx_pre, len(idx_ictal), replace=False)
    idx_sel = np.hstack([idx_ictal, idx_pre_down])

    X_train_bal = X_train[idx_sel]
    Y_train_bal = Y_train[idx_sel]

    # ---------- MODELLI ----------
    clf_lr = LogisticRegression(max_iter=1000)
    clf_gb = GradientBoostingClassifier(n_estimators=200, max_depth=3)

    clf_lr.fit(X_train_bal, Y_train_bal)
    clf_gb.fit(X_train_bal, Y_train_bal)

    Y_prob_lr = clf_lr.predict_proba(X_test)[:, 1]
    Y_pred_lr = (Y_prob_lr > 0.5).astype(int)

    Y_prob_gb = clf_gb.predict_proba(X_test)[:, 1]
    Y_pred_gb = (Y_prob_gb > 0.5).astype(int)

    # ---------- METRICHE ----------
    def compute_metrics(y_true, y_pred, y_prob):
        auc = roc_auc_score(y_true, y_prob)
        acc = accuracy_score(y_true, y_pred)
        bal = balanced_accuracy_score(y_true, y_pred)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        return auc, acc, bal, sens, spec

    for model, yp, yprob in [
        ('LogisticRegression', Y_pred_lr, Y_prob_lr),
        ('GradientBoosting', Y_pred_gb, Y_prob_gb)
    ]:
        auc, acc, bal, sens, spec = compute_metrics(Y_test, yp, yprob)
        results.append({
            'patient': p,
            'model': model,
            'AUC': auc,
            'accuracy': acc,
            'balanced_accuracy': bal,
            'sensitivity': sens,
            'specificity': spec
        })


In [25]:
df_results = pd.DataFrame(results)

print(
    df_results
    .groupby('model')[['AUC','accuracy','balanced_accuracy','sensitivity','specificity']]
    .mean()
)


                         AUC  accuracy  balanced_accuracy  sensitivity  \
model                                                                    
GradientBoosting    0.919497  0.888384           0.853879     0.808333   
LogisticRegression  0.832034  0.840404           0.784824     0.711458   

                    specificity  
model                            
GradientBoosting       0.899425  
LogisticRegression     0.858190  
